# Example 2: Box Stokes flow

In [ ]:
from pathlib import Path
import shutil

from mpi4py import MPI
from petsc4py import PETSc

import numpy as np
import ufl
import basix.ufl
import dolfinx
from dolfinx import fem
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
from dolfinx.io import XDMFFile, VTXWriter
import io4dolfinx

COMM = MPI.COMM_WORLD
CHECKPOINT = Path("mwe_3/sim_checkpoint.bp")

INLET_ID = 7
OUTLET_ID = 8
WALLS_ID = 9


def cg2_vec(msh):
    el = basix.ufl.element(
        "Lagrange", msh.topology.cell_name(), 2, shape=(msh.geometry.dim,)
    )
    return fem.functionspace(msh, el)


def load_from_xdmf():
    with XDMFFile(COMM, "mwe_3/mesh_data/mesh_domains.xdmf", "r") as f:
        msh = f.read_mesh(name="Grid")
    msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
    msh.topology.create_connectivity(msh.topology.dim - 1, msh.topology.dim)
    with XDMFFile(COMM, "mwe_3/mesh_data/mesh_boundaries.xdmf", "r") as f:
        facet_mt = f.read_meshtags(msh, name="Grid")
    return msh, facet_mt


def load_from_checkpoint():
    msh = io4dolfinx.read_mesh(CHECKPOINT, COMM)
    msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
    msh.topology.create_connectivity(msh.topology.dim - 1, msh.topology.dim)
    facet_mt = io4dolfinx.read_meshtags(CHECKPOINT, msh, meshtag_name="facets")
    w = fem.Function(cg2_vec(msh), name="velocity")
    io4dolfinx.read_function(CHECKPOINT, w, time=0.0, name="velocity")
    return msh, facet_mt, w


def solve_stokes(msh, facet_mt):
    """Taylor-Hood Stokes with plug inlet, no-slip walls, natural outflow.

    Writes mesh + facet tags + velocity together to a fresh checkpoint.
    """
    gdim = msh.geometry.dim
    fdim = msh.topology.dim - 1
    P2 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 2, shape=(gdim,))
    P1 = basix.ufl.element("Lagrange", msh.topology.cell_name(), 1)
    W = fem.functionspace(msh, basix.ufl.mixed_element([P2, P1]))
    V_sub, _ = W.sub(0).collapse()

    u_in = fem.Function(V_sub)
    u_in.interpolate(
        lambda x: np.vstack([0.1 * np.ones(x.shape[1]), np.zeros(x.shape[1])])
    )
    bcs = [
        fem.dirichletbc(
            u_in,
            fem.locate_dofs_topological(
                (W.sub(0), V_sub), fdim, facet_mt.find(INLET_ID)
            ),
            W.sub(0),
        ),
        fem.dirichletbc(
            fem.Function(V_sub),
            fem.locate_dofs_topological(
                (W.sub(0), V_sub), fdim, facet_mt.find(WALLS_ID)
            ),
            W.sub(0),
        ),
    ]

    u, p = ufl.TrialFunctions(W)
    v, q = ufl.TestFunctions(W)
    mu = fem.Constant(msh, PETSc.ScalarType(1.0))
    a = (
        mu * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
        - ufl.inner(p, ufl.div(v)) * ufl.dx
        - ufl.inner(ufl.div(u), q) * ufl.dx
    )
    L = ufl.inner(fem.Constant(msh, PETSc.ScalarType((0.0,) * gdim)), v) * ufl.dx

    problem = LinearProblem(
        a,
        L,
        bcs=bcs,
        petsc_options_prefix="stokes",
        petsc_options={
            "ksp_type": "preonly",
            "pc_type": "lu",
            "pc_factor_mat_solver_type": "mumps",
            "ksp_error_if_not_converged": True,
        },
    )
    wh = problem.solve()

    u_h = fem.Function(cg2_vec(msh), name="velocity")
    u_h.interpolate(wh.sub(0).collapse())
    u_h.x.scatter_forward()

    io4dolfinx.write_mesh(CHECKPOINT, msh)
    io4dolfinx.write_meshtags(CHECKPOINT, msh, facet_mt, meshtag_name="facets")
    io4dolfinx.write_function(CHECKPOINT, u_h, time=0.0, name="velocity")

    return u_h


def advection_diffusion(msh, facet_mt, w, D_value=1e-5):
    """SIPG DG1 advection-diffusion with given velocity ``w``."""
    V = fem.functionspace(msh, ("DG", 1))
    u = fem.Function(V)
    v = ufl.TestFunction(V)

    n = ufl.FacetNormal(msh)
    h = ufl.CellDiameter(msh)
    ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_mt)
    dS, dx = ufl.dS, ufl.dx

    D = fem.Constant(msh, PETSc.ScalarType(D_value))
    penalty = fem.Constant(msh, PETSc.ScalarType(10))
    u_inlet = fem.Constant(msh, PETSc.ScalarType(0.0))
    f_source = fem.Constant(msh, PETSc.ScalarType(1.0))
    lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

    F = -ufl.inner(w * u, ufl.grad(v)) * dx
    F += ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v, n)) * dS

    F_inlet_adv = -ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_inlet, v) * ds(INLET_ID)
    F_outlet_surf = ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds(OUTLET_ID)
    F += F_inlet_adv + F_outlet_surf

    F += D * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
    F += -D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v, n)) * dS
    F += -D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v))) * dS
    F += D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v, n)) * dS

    F_inlet_nitsche = D * (
        -ufl.inner(ufl.grad(u), v * n) * ds(INLET_ID)
        - ufl.inner(ufl.grad(v), (u - u_inlet) * n) * ds(INLET_ID)
        + (penalty / h) * ufl.inner(u - u_inlet, v) * ds(INLET_ID)
    )
    F += F_inlet_nitsche

    F_inlet_terms = F_inlet_adv + F_inlet_nitsche

    F += -ufl.inner(f_source, v) * dx

    problem = NonlinearProblem(
        F,
        u,
        J=ufl.derivative(F, u),
        petsc_options_prefix="adv_diff",
        petsc_options={
            "snes_atol": 1e-12,
            "snes_rtol": 1e-12,
            "snes_max_it": 30,
            "ksp_type": "preonly",
            "pc_type": "lu",
            "pc_factor_mat_solver_type": "mumps",
        },
    )
    u = problem.solve()
    u.x.scatter_forward()

    writer = VTXWriter(COMM, "mwe_box.bp", u, "BP5")
    writer.write(t=0.0)

    tdim = msh.topology.dim
    fdim = tdim - 1
    msh.topology.create_connectivity(fdim, tdim)
    msh.topology.create_connectivity(tdim, tdim)
    owned_size = V.dofmap.index_map.size_local * V.dofmap.index_map_bs

    def get_owned_dofs(marker):
        facets = facet_mt.find(marker)
        f_to_c = msh.topology.connectivity(fdim, tdim)
        cells = np.unique(np.concatenate([f_to_c.links(f) for f in facets]))
        dofs = fem.locate_dofs_topological(V, tdim, cells)
        return dofs[dofs < owned_size]

    inlet_dofs = get_owned_dofs(INLET_ID)
    outlet_dofs = get_owned_dofs(OUTLET_ID)
    wall_dofs = get_owned_dofs(WALLS_ID)

    def compute_consistent_flux(residual_form: fem.Form, dofs: np.ndarray):
        residual = fem.assemble_vector(residual_form)
        residual.scatter_reverse(dolfinx.la.InsertMode.add)
        residual.scatter_forward()
        local_flux = np.sum(residual.array[dofs])
        return msh.comm.allreduce(local_flux, op=MPI.SUM)

    F_no_outlet = fem.form(F - F_outlet_surf)
    F_no_inlet = fem.form(F - F_inlet_terms)
    F_form = fem.form(F)

    flux_outlet = -compute_consistent_flux(F_no_outlet, outlet_dofs)
    flux_inlet = -compute_consistent_flux(F_no_inlet, inlet_dofs)
    flux_wall = compute_consistent_flux(F_form, wall_dofs)

    source_total = msh.comm.allreduce(
        fem.assemble_scalar(fem.form(f_source * dx)), op=MPI.SUM
    )

    consist_total = flux_outlet + flux_wall + flux_inlet

    def pct(val):
        return 100 * val / source_total

    print(f"Source integral : {source_total:.6e}")
    print()
    print(f"{'Flux outlet':20s} {flux_outlet:14.6e}")
    print(f"{'Flux inlet':20s} {flux_inlet:14.6e}")
    print(f"{'Flux wall':20s} {flux_wall:14.6e}")
    print()
    c_bal = source_total - consist_total
    print(f"{'Total flux':20s} {consist_total:14.6e}")
    print(f"{'Balance residual':20s} {c_bal:14.6e} ({pct(c_bal):+.2f}%)")


if __name__ == "__main__":
    msh_xdmf, facet_mt_xdmf = load_from_xdmf()
    u_stokes = solve_stokes(msh_xdmf, facet_mt_xdmf)

    msh_ck, facet_mt_ck, w_ck = load_from_checkpoint()
    advection_diffusion(msh_ck, facet_mt_ck, w_ck)


Source integral : 1.080000e+00

Flux outlet            1.079979e+00
Flux inlet             2.121287e-05
Flux wall              1.477355e-16

Total flux             1.080000e+00
Balance residual       4.218847e-15 (+0.00%)
